# Coordinate Descent for Logistic Regression

This notebook implements **coordinate descent** (CD) for binary logistic regression and compares two coordinate-selection strategies: greedy (largest-gradient) and random. We benchmark both against scikit-learn's SAG solver to illustrate convergence behavior. By the end you will understand the CD update rule, the impact of coordinate selection, and when CD is advantageous in practice.

> **Requirements:** `numpy`, `matplotlib`, `scikit-learn` (install via `pip install numpy matplotlib scikit-learn` if needed).

In [ ]:
import numpy as np
from tqdm import tqdm
from random import randint
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression




## What is Coordinate Descent?

Coordinate descent minimizes a function $f(x_1, \ldots, x_n)$ by updating **one coordinate at a time** while keeping all others fixed. At iteration $k$ the generic update is

$$
x_i^{(k+1)} = \arg\min_{x_i} \; f\bigl(x_1^{(k)}, \ldots, x_{i-1}^{(k)}, \; x_i, \; x_{i+1}^{(k)}, \ldots, x_n^{(k)}\bigr).
$$

When a closed-form minimizer is unavailable (as in logistic regression), we replace the exact minimization with a gradient step along coordinate $i$:

$$
w_i^{(k+1)} = w_i^{(k)} - \eta \, \frac{\partial L(w^{(k)})}{\partial w_i}.
$$

### Coordinate-selection strategies

| Strategy | Rule | Cost per iteration |
|---|---|---|
| **Cyclic** | Cycle through $i = 1, 2, \ldots, n, 1, 2, \ldots$ | $O(n)$ for one full pass |
| **Random** | Pick $i$ uniformly at random | $O(1)$ per update |
| **Greedy (Gauss-Southwell)** | Pick $i = \arg\max_j |\partial L / \partial w_j|$ | $O(n)$ to scan all gradients |

### When is coordinate descent advantageous?

- **Separable regularizers.** If $f(x) = g(x) + \sum_i r_i(x_i)$ (e.g., $\ell_1$-regularization), the per-coordinate subproblem decouples and often has a closed-form solution (soft thresholding).
- **Cheap per-coordinate updates.** When updating one coordinate is much cheaper than a full gradient step (e.g., sparse data, large $n$).
- **Large-scale problems.** CD has been highly successful for training SVMs (SMO algorithm) and for $\ell_1$-regularized regression (LASSO via pathwise CD).

### Experiment overview

We apply coordinate descent to **binary logistic regression** on the UCI Wine dataset (classes 1 vs. 2). Two selection strategies are compared:

1. **Greedy (best-coordinate):** at each step, update the weight whose partial derivative has the largest absolute value.
2. **Random:** at each step, pick a coordinate uniformly at random.

Both are benchmarked against scikit-learn's `LogisticRegression` (SAG solver) to obtain a reference loss $L^*$.

> **Caveat:** Coordinate descent can get stuck at non-stationary points for nonsmooth objectives whose level sets have axis-aligned corners. For smooth, convex objectives like logistic regression this is not an issue.

## Loss Function and Gradients

We use the **cross-entropy (log) loss**. Let $\widehat{y}_i = \sigma(w^\top x_i)$ denote the predicted probability for sample $i$, where $\sigma$ is the sigmoid function. The loss is

$$
L(w) = -\frac{1}{n}\sum_{i=1}^{n} \bigl[ y_i \log \widehat{y}_i + (1 - y_i) \log(1 - \widehat{y}_i) \bigr].
$$

The partial derivative with respect to weight $w_j$ is

$$
\frac{\partial L(w)}{\partial w_j} = \frac{1}{n}\sum_{i=1}^{n} (\widehat{y}_i - y_i)\, x_{ij}.
$$

The coordinate descent procedure then repeats:

1. Compute predictions $\widehat{y} = \sigma(Xw)$.
2. Compute the full gradient $\nabla L(w)$.
3. Select a coordinate $j$ (greedy or random).
4. Update $w_j \leftarrow w_j - \eta \, \frac{\partial L}{\partial w_j}$.
5. Repeat until convergence.

## Data Preparation

We load the Wine dataset, restrict to a binary classification task (classes 1 vs. 2), normalize features to zero mean and unit range, and append a bias column of ones.

In [ ]:
from sklearn.datasets import load_wine

# Load the wine dataset
wine = load_wine()

# Features and target
X, y = wine.data, wine.target

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

# Transform into binary classification: class 1 vs class 2
idxs = [i for i in range(len(y)) if y[i] == 1 or y[i] == 2]
X, y = X[idxs], y[idxs]

# Normalize features to zero mean, unit range
X = (X - X.mean(axis=0)) / (X.max(axis=0) - X.min(axis=0))
# Append bias column
X = np.hstack((X, np.ones(len(X)).reshape(len(X), 1)))

# Relabel: class 1 -> 0, class 2 -> 1
y = np.array(list(map(lambda x: 0 if x == 1 else 1, y)))

### Reference solution via scikit-learn

We fit an (essentially) unregularized logistic regression with scikit-learn's SAG solver to obtain the optimal loss $L^*$ as a convergence target.

In [ ]:
# Scikit-learn logistic regression (our benchmark)
reg = LogisticRegression(solver='sag', C=100000, max_iter=10000).fit(X, y)
L_star = log_loss(y, reg.predict_proba(X))

print("Reference loss L* = {:<16f}".format(L_star))

### Helper functions

Below we define `sigmoid`, `cross_entropy_loss`, and `cross_entropy_grad` -- the building blocks for the coordinate descent loop.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def cross_entropy_loss(y, y_pred):
    """Binary cross-entropy loss."""
    loss = -(1 / len(y)) * np.sum(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
    return loss

def cross_entropy_grad(y, y_pred, x):
    """Gradient of cross-entropy w.r.t. each weight."""
    return list(np.dot((y_pred - y), X)[0])

## Greedy Coordinate Descent (Gauss-Southwell Rule)

At each iteration we select the coordinate with the largest gradient magnitude:

$$
j^* = \arg\max_{j} \left|\frac{\partial L(w)}{\partial w_j}\right|,
$$

and update only $w_{j^*}$. This **Gauss-Southwell** rule guarantees at least as much per-step decrease as any other single-coordinate choice, making it the theoretically optimal greedy strategy.

In [ ]:
# Greedy (best-coordinate) descent
w = np.zeros(14).reshape(14, 1)

eta = 0.01
loss = []
num_iter = 100000

for t in tqdm(range(num_iter)):
    y_pred = sigmoid(np.dot(w.T, X.T))
    grad = cross_entropy_grad(y, y_pred, X)

    largest = np.argmax(np.abs(grad))  # coordinate with largest |gradient|
    loss.append(cross_entropy_loss(y, y_pred))

    # Update only the selected coordinate
    w[largest] = w[largest] - eta * grad[largest]

y_pred = sigmoid(np.dot(w.T, X.T))
y_pred = np.array(list(map(lambda x: 1 if x >= 0.5 else 0, y_pred.flatten())))

print("Accuracy (greedy CD): {0}".format(accuracy_score(y, y_pred)))
print("Final loss: {0}".format(loss[-1]))

## Randomized Coordinate Descent

Here we pick the coordinate $j$ **uniformly at random** from $\{1, \ldots, n\}$ at each step. Randomized CD is simpler and avoids the $O(n)$ cost of scanning all gradients. In expectation, its convergence rate is $O(n/k)$ for convex objectives, compared to $O(1/k)$ for full gradient descent -- the factor of $n$ reflects that each step only touches one coordinate.

In [ ]:
# Random coordinate descent
w = np.zeros(14).reshape(14, 1)
w_rand = w

eta = 0.01
loss_rand = []
num_iter = 100000

for t in tqdm(range(num_iter)):
    y_pred = sigmoid(np.dot(w.T, X.T))
    loss_rand.append(cross_entropy_loss(y, y_pred))
    grad = cross_entropy_grad(y, y_pred, X)
    random_idx = randint(0, 13)  # pick a random coordinate

    # Update only the randomly selected coordinate
    w_rand[random_idx] = w_rand[random_idx] - eta * grad[random_idx]

y_pred = sigmoid(np.dot(w_rand.T, X.T))
y_pred = np.array(list(map(lambda x: 1 if x >= 0.5 else 0, y_pred.flatten())))

print("Accuracy (random CD): {0}".format(accuracy_score(y, y_pred)))

## Convergence Comparison

The plot below shows the cross-entropy loss versus iteration count for both strategies, with the reference loss $L^*$ from scikit-learn shown as a horizontal line.

In [ ]:
plt.figure(figsize=(10, 10))
plt.plot(loss, 'g-', label='Greedy (Best Coordinate) Descent')
plt.plot(loss_rand, 'r-', label='Random Coordinate Descent')
plt.axhline(y=L_star, color='y', linestyle='--', label='$L^*$ (scikit-learn)')
plt.title('Iteration vs. Loss', fontsize=25)
plt.xlabel('Iteration', fontsize=15)
plt.xlim(0, 100000)
plt.ylim((-0.01, 0.04))
plt.ylabel('Cross-Entropy Loss', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid()
plt.legend(fontsize=14)
plt.show()

## Summary

**Key takeaways from this notebook:**

- **Coordinate descent** updates one variable at a time: $w_j \leftarrow w_j - \eta\,\partial L / \partial w_j$. It is a simple yet effective optimization strategy, especially for large-scale and separable problems.
- **Greedy (Gauss-Southwell) selection** converges significantly faster than random selection on this problem, because it always makes the most impactful single-coordinate update.
- **Randomized selection** is cheaper per iteration (no need to scan all gradients) and still converges to $L^*$, but requires many more iterations.
- Both strategies eventually reach the same optimal loss $L^* \approx 0.0004$, confirming that coordinate descent converges to the global minimizer for this convex problem.
- **Practical considerations:** Coordinate descent is especially attractive when (a) the objective has separable structure (e.g., $\ell_1$-regularized problems where soft-thresholding gives closed-form per-coordinate updates), or (b) the number of features is very large and full gradient computation is expensive.